# 04f - Volatility-Granger Graph Experiment

This notebook tests a separate Granger graph version based on weekly realized-volatility predictability. It does not overwrite the frozen return-Granger artifacts.

Primary artifacts:

- `data/graphs/granger_vol_pvalues.parquet`
- `data/graphs/granger_vol_edges.parquet`
- `data/graphs/granger_vol_meta.json`
- `data/results/checkpoints/gnn_granger_vol_best.pt`
- `data/results/val_preds_gnn_granger_vol.parquet`
- `data/results/test_preds_gnn_granger_vol.parquet`

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config
from src.evaluate import compute_all_ranking_metrics, compute_metrics
from src.graphs import build_volatility_granger_graph, run_volatility_granger_tests
from src.models import GNNModel
from src.train import predict_gnn_split, train_gnn_volatility_granger

pd.set_option('display.max_columns', 40)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'ROOT={ROOT}')
print(f'device={device}')

ROOT=C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project_granger_vol
device=cuda


## Run Controls

The all-pairs volatility-Granger p-value computation can be expensive. Leave `RUN_HEAVY_GRAPH_BUILD=False` to inspect existing artifacts only. Set it to `True` when you are ready to generate the graph in this notebook.

In [2]:
RUN_HEAVY_GRAPH_BUILD = False
RUN_TRAINING = True

USE_GPU = None      # None = auto, True = force CUDA, False = CPU
N_WORKERS = None    # CPU worker count when USE_GPU is False or CUDA is unavailable
MAX_EPOCHS = config.GNN_MAX_EPOCHS

print({
    'RUN_HEAVY_GRAPH_BUILD': RUN_HEAVY_GRAPH_BUILD,
    'RUN_TRAINING': RUN_TRAINING,
    'USE_GPU': USE_GPU,
    'N_WORKERS': N_WORKERS,
    'MAX_EPOCHS': MAX_EPOCHS,
})

{'RUN_HEAVY_GRAPH_BUILD': False, 'RUN_TRAINING': True, 'USE_GPU': None, 'N_WORKERS': None, 'MAX_EPOCHS': 150}


In [3]:
raw_dir = Path(config.DATA_RAW_DIR)
features_dir = Path(config.DATA_FEATURES_DIR)
graphs_dir = Path(config.DATA_GRAPHS_DIR)
results_dir = Path(config.DATA_RESULTS_DIR)
ckpt_dir = Path(config.CHECKPOINTS_DIR)

required = {
    'log_returns': raw_dir / 'log_returns.parquet',
    'target': features_dir / 'target.parquet',
    'splits': features_dir / 'splits.parquet',
    'features': features_dir / 'features.parquet',
}
status = pd.DataFrame([
    {'artifact': name, 'path': str(path.relative_to(ROOT)), 'exists': path.exists()}
    for name, path in required.items()
])
display(status)
missing = status.loc[~status['exists'], 'artifact'].tolist()
if missing:
    raise FileNotFoundError(f'Missing required input artifacts: {missing}')

,artifact,path,exists
0,log_returns,data\raw\log_returns.parquet,True
1,target,data\features\target.parquet,True
2,splits,data\features\splits.parquet,True
3,features,data\features\features.parquet,True


In [4]:
log_returns = pd.read_parquet(required['log_returns'])
target = pd.read_parquet(required['target'])
target.index = pd.to_datetime(target.index)
splits = pd.read_parquet(required['splits'])
splits['week'] = pd.to_datetime(splits['week'])

tickers = target.columns.tolist()
log_returns = log_returns.reindex(columns=tickers)

feat_df = pd.read_parquet(required['features'])
feat_df['week'] = pd.to_datetime(feat_df['week'])
feat_df = feat_df[feat_df['week'].isin(target.index)].copy()
feature_cols = [c for c in feat_df.columns if c not in {'week', 'ticker'}]

ticker_order = {ticker: i for i, ticker in enumerate(tickers)}
feat_df['_ord'] = feat_df['ticker'].map(ticker_order)
feat_df = feat_df.sort_values(['week', '_ord'])

n_weeks, n_stocks, n_feats = target.shape[0], target.shape[1], len(feature_cols)
features_3d = feat_df[feature_cols].to_numpy(dtype=float).reshape(n_weeks, n_stocks, n_feats)
target_arr = target.to_numpy(dtype=float)

assert features_3d.shape == (n_weeks, n_stocks, n_feats)
assert target_arr.shape == (n_weeks, n_stocks)
print(f'log_returns: {log_returns.shape}')
print(f'features_3d: {features_3d.shape}')
print(f'target: {target.shape}')
print(f'splits: {splits["split"].value_counts().to_dict()}')

log_returns: (2764, 465)
features_3d: (572, 465, 10)
target: (572, 465)
splits: {'train': 417, 'test': 103, 'val': 52}


## Build or Load Volatility-Granger Graph

In [5]:
pval_path = graphs_dir / config.GRANGER_VOL_PVALUES_FILE
edge_path = graphs_dir / config.GRANGER_VOL_EDGES_FILE
meta_path = graphs_dir / config.GRANGER_VOL_META_FILE

artifact_status = pd.DataFrame([
    {'artifact': 'vol_granger_pvalues', 'path': str(pval_path.relative_to(ROOT)), 'exists': pval_path.exists()},
    {'artifact': 'vol_granger_edges', 'path': str(edge_path.relative_to(ROOT)), 'exists': edge_path.exists()},
    {'artifact': 'vol_granger_meta', 'path': str(meta_path.relative_to(ROOT)), 'exists': meta_path.exists()},
])
display(artifact_status)

if RUN_HEAVY_GRAPH_BUILD and not pval_path.exists():
    run_volatility_granger_tests(
        log_returns=log_returns,
        tickers=tickers,
        lag=config.GRANGER_VOL_LAG,
        use_gpu=USE_GPU,
        n_workers=N_WORKERS,
    )
elif not pval_path.exists():
    print('Volatility-Granger p-values are missing. Set RUN_HEAVY_GRAPH_BUILD=True to generate them here.')

if pval_path.exists() and (RUN_HEAVY_GRAPH_BUILD or not edge_path.exists()):
    granger_vol_edge_index, correction_used = build_volatility_granger_graph(tickers)
elif edge_path.exists():
    edge_df = pd.read_parquet(edge_path)
    granger_vol_edge_index = torch.tensor(edge_df[['src', 'dst']].values.T, dtype=torch.long)
    correction_used = json.load(open(meta_path)).get('correction_used', 'unknown') if meta_path.exists() else 'unknown'
else:
    granger_vol_edge_index = None
    correction_used = None

print(f'correction_used={correction_used}')
print(f'edge_index={None if granger_vol_edge_index is None else tuple(granger_vol_edge_index.shape)}')

,artifact,path,exists
0,vol_granger_pvalues,data\graphs\granger_vol_pvalues.parquet,True
1,vol_granger_edges,data\graphs\granger_vol_edges.parquet,True
2,vol_granger_meta,data\graphs\granger_vol_meta.json,True


correction_used=bonferroni
edge_index=(2, 26169)


In [6]:
rows = []
for label, path, directed in [
    ('Return-Granger frozen', graphs_dir / 'granger_edges.parquet', True),
    ('Volatility-Granger', edge_path, True),
]:
    if path.exists():
        df = pd.read_parquet(path)
        n_edges = len(df)
        n_possible = len(tickers) * (len(tickers) - 1)
        rows.append({
            'graph': label,
            'edges': n_edges,
            'density': n_edges / n_possible,
            'mean_out_degree': n_edges / len(tickers),
            'path': str(path.relative_to(ROOT)),
        })
    else:
        rows.append({'graph': label, 'edges': np.nan, 'density': np.nan, 'mean_out_degree': np.nan, 'path': str(path.relative_to(ROOT))})

graph_summary = pd.DataFrame(rows)
display(graph_summary)
if meta_path.exists():
    display(pd.Series(json.load(open(meta_path)), name='volatility_granger_meta').to_frame())

,graph,edges,density,mean_out_degree,path
0,Return-Granger frozen,13886,0.064359,29.862366,data\graphs\granger_edges.parquet
1,Volatility-Granger,26169,0.121288,56.277419,data\graphs\granger_vol_edges.parquet


,volatility_granger_meta
correction_used,bonferroni
n_edges,26169
n_pairs,215760
signal_label,weekly realized volatility
pvalue_file,granger_vol_pvalues.parquet
edges_file,granger_vol_edges.parquet


## Train Experimental Model

In [7]:
ckpt_path = ckpt_dir / 'gnn_granger_vol_best.pt'
val_loss_path = results_dir / 'gnn_granger_vol_val_loss.json'

model = None
if granger_vol_edge_index is None:
    print('Skipping training: volatility-Granger graph is not available.')
elif RUN_TRAINING:
    model, val_loss_history = train_gnn_volatility_granger(
        features=features_3d,
        target=target_arr,
        week_index=target.index,
        granger_vol_edge_index=granger_vol_edge_index,
        splits=splits,
        device=device,
        max_epochs=MAX_EPOCHS,
    )
elif ckpt_path.exists():
    model = GNNModel(in_channels=n_feats).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()
    print(f'Loaded existing checkpoint: {ckpt_path.relative_to(ROOT)}')
else:
    print('No checkpoint found. Set RUN_TRAINING=True after the volatility graph exists.')

if val_loss_path.exists():
    val_loss = json.load(open(val_loss_path))
    print(f"best_val_loss={val_loss.get('best_val_loss')}")

C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project_granger_vol\.venv\lib\site-packages\torch\optim\lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project_granger_vol\src\models.py:289: UserWarning: GNNModel.forward: input features contain NaN values. Imputing with 0 (z-score mean). Expected during warm-up weeks.
  warnings.warn(


Epoch   1  train=0.040787  val=0.024482


  Epoch 2  step 417/417

Epoch   2  train=0.037638  val=0.023625


...

Epoch  12  train=0.033759  val=0.023507


Epoch  13  train=0.032601  val=0.025598


Epoch  14  train=0.031395  val=0.024605


Epoch  15  train=0.030451  val=0.023984


Epoch  16  train=0.029660  val=0.024214
Early stop at epoch 16 (no improvement for 10 epochs)
best_val_loss=0.02272586519113527


## Prediction Artifacts

In [8]:
val_pred_path = results_dir / 'val_preds_gnn_granger_vol.parquet'
test_pred_path = results_dir / 'test_preds_gnn_granger_vol.parquet'

if model is not None and granger_vol_edge_index is not None:
    edge_fn = lambda week: granger_vol_edge_index
    val_preds = predict_gnn_split(model, features_3d, target.index, edge_fn, splits, tickers, device, split_name='val')
    test_preds = predict_gnn_split(model, features_3d, target.index, edge_fn, splits, tickers, device, split_name='test')
    val_preds.to_parquet(val_pred_path)
    test_preds.to_parquet(test_pred_path)
    print(f'Saved {val_pred_path.relative_to(ROOT)} shape={val_preds.shape}')
    print(f'Saved {test_pred_path.relative_to(ROOT)} shape={test_preds.shape}')
elif test_pred_path.exists():
    print(f'Prediction artifact exists: {test_pred_path.relative_to(ROOT)}')
else:
    print('Prediction artifacts are not available yet.')

Saved data\results\val_preds_gnn_granger_vol.parquet shape=(52, 465)
Saved data\results\test_preds_gnn_granger_vol.parquet shape=(103, 465)


## Evaluation Snapshot

In [9]:
prediction_files = {
    'GNN-Granger-Volatility': test_pred_path,
    'GNN-Granger': results_dir / 'test_preds_gnn_granger.parquet',
    'GNN-Granger + Macro': results_dir / 'test_preds_gnn_granger_macro.parquet',
    'GNN-Correlation': results_dir / 'test_preds_gnn_corr.parquet',
    'LSTM': results_dir / 'test_preds_lstm.parquet',
}

test_weeks = splits.loc[splits['split'] == 'test', 'week']
target_test = target.reindex(test_weeks)

metric_rows = []
rank_rows = []
for model_name, path in prediction_files.items():
    if not path.exists():
        continue
    preds = pd.read_parquet(path).reindex(index=target_test.index, columns=tickers)
    metrics = compute_metrics(target_test.to_numpy(dtype=float), preds.to_numpy(dtype=float))
    metric_rows.append({'model': model_name, **metrics, 'prediction_path': str(path.relative_to(ROOT))})
    ranking = compute_all_ranking_metrics(preds, target_test, model_name=model_name)
    rank_rows.append({
        'model': model_name,
        'mean_ic': ranking.mean_ic,
        'icir': ranking.icir,
        'pct_positive_ic': ranking.pct_positive_ic,
        'mean_hit_rate': ranking.mean_hit_rate,
        'mean_pairwise_acc': ranking.mean_pairwise_acc,
    })

ml_metrics = pd.DataFrame(metric_rows).sort_values('mse') if metric_rows else pd.DataFrame()
rank_metrics = pd.DataFrame(rank_rows).sort_values('mean_ic', ascending=False) if rank_rows else pd.DataFrame()
display(ml_metrics)
display(rank_metrics)

if not test_pred_path.exists():
    print('GNN-Granger-Volatility is not evaluated yet because its test prediction artifact is missing.')

,model,mse,mae,r2,prediction_path
2,GNN-Granger + Macro,0.031439,0.107706,0.168248,data\results\test_preds_gnn_granger_macro.parquet
3,GNN-Correlation,0.032191,0.107638,0.148354,data\results\test_preds_gnn_corr.parquet
4,LSTM,0.032424,0.109583,0.142188,data\results\test_preds_lstm.parquet
0,GNN-Granger-Volatility,0.033080,0.120786,0.124823,data\results\test_preds_gnn_granger_vol.parquet
1,GNN-Granger,0.033702,0.119049,0.108378,data\results\test_preds_gnn_granger.parquet


,model,mean_ic,icir,pct_positive_ic,mean_hit_rate,mean_pairwise_acc
2,GNN-Granger + Macro,0.429478,4.064242,1.0,0.493555,0.648191
4,LSTM,0.428819,4.362584,1.0,0.498996,0.648435
3,GNN-Correlation,0.416509,3.439945,1.0,0.491881,0.644850
0,GNN-Granger-Volatility,0.410440,4.387533,1.0,0.489454,0.608993
1,GNN-Granger,0.374945,3.663005,1.0,0.472631,0.629316


## Registry Row Preview

In [10]:
graph_version = f"granger_vol_weekly_rv_lag_{config.GRANGER_VOL_LAG}_{correction_used or 'unknown'}"
registry_preview = pd.DataFrame([{
    'experiment_id': 'gnn_granger_vol',
    'model_name': 'GNN-Granger-Volatility',
    'model_family': 'GNN',
    'graph_type': 'granger_volatility',
    'loss_type': 'mse',
    'feature_version': 'stock_features_v1',
    'graph_version': graph_version,
    'checkpoint_path': 'data/results/checkpoints/gnn_granger_vol_best.pt',
    'validation_metrics_path': 'data/results/gnn_granger_vol_val_loss.json',
    'test_metrics_path': '',
    'prediction_path': 'data/results/test_preds_gnn_granger_vol.parquet',
    'notes': 'Experimental static Granger graph based on weekly realized-volatility Granger tests over train data only.',
}])
display(registry_preview)

,experiment_id,model_name,model_family,graph_type,loss_type,feature_version,graph_version,checkpoint_path,validation_metrics_path,test_metrics_path,prediction_path,notes
0,gnn_granger_vol,GNN-Granger-Volatility,GNN,granger_volatility,mse,stock_features_v1,granger_vol_weekly_rv_lag_5_bonferroni,data/results/checkpoints/gnn_granger_vol_best.pt,data/results/gnn_granger_vol_val_loss.json,,data/results/test_preds_gnn_granger_vol.parquet,Experimental static Granger graph based on wee...
